In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

from sklearn.preprocessing import OneHotEncoder

In [3]:
data = pd.read_csv("Agriculture_price_dataset.csv")

print("Dataset shape:", data.shape)
print(data.head())

Dataset shape: (737392, 10)
           STATE District Name        Market Name Commodity           Variety  \
0    Maharashtra        nashik  Lasalgaon(Niphad)     Wheat  Maharashtra 2189   
1    Maharashtra        satara              Patan    Tomato             Other   
2  Uttar Pradesh      mainpuri              Bewar    Potato             Local   
3      Rajasthan   chittorgarh          Nimbahera     Wheat             Other   
4      Rajasthan    pratapgarh         Pratapgarh     Onion             Other   

  Grade  Min_Price  Max_Price  Modal_Price Price Date  
0   FAQ     2172.0     2399.0       2300.0   6/6/2023  
1   FAQ     1000.0     1500.0       1250.0   6/6/2023  
2   FAQ      800.0      820.0        810.0   6/6/2023  
3   FAQ     2040.0     2668.0       2300.0   6/6/2023  
4   FAQ      476.0     1043.0        617.0   6/6/2023  


In [4]:
# select only required columns
data = data[['STATE','District Name','Market Name','Commodity',
             'Price Date','Min_Price','Max_Price','Modal_Price']]

# rename columns for easier coding
data.columns = [
    'state',
    'district',
    'market',
    'commodity',
    'date',
    'min_price',
    'max_price',
    'modal_price'
]

print(data.head())
print(data.columns)

           state     district             market commodity      date  \
0    Maharashtra       nashik  Lasalgaon(Niphad)     Wheat  6/6/2023   
1    Maharashtra       satara              Patan    Tomato  6/6/2023   
2  Uttar Pradesh     mainpuri              Bewar    Potato  6/6/2023   
3      Rajasthan  chittorgarh          Nimbahera     Wheat  6/6/2023   
4      Rajasthan   pratapgarh         Pratapgarh     Onion  6/6/2023   

   min_price  max_price  modal_price  
0     2172.0     2399.0       2300.0  
1     1000.0     1500.0       1250.0  
2      800.0      820.0        810.0  
3     2040.0     2668.0       2300.0  
4      476.0     1043.0        617.0  
Index(['state', 'district', 'market', 'commodity', 'date', 'min_price',
       'max_price', 'modal_price'],
      dtype='object')


In [5]:
# convert date column into datetime
data['date'] = pd.to_datetime(data['date'])

# extract year, month, day
data['year'] = data['date'].dt.year
data['month'] = data['date'].dt.month
data['day'] = data['date'].dt.day

print(data[['date','year','month','day']].head())

        date  year  month  day
0 2023-06-06  2023      6    6
1 2023-06-06  2023      6    6
2 2023-06-06  2023      6    6
3 2023-06-06  2023      6    6
4 2023-06-06  2023      6    6


In [6]:
# create new price related features
data['price_spread'] = data['max_price'] - data['min_price']

data['avg_price'] = (data['max_price'] + data['min_price']) / 2

print(data[['min_price','max_price','price_spread','avg_price']].head())

   min_price  max_price  price_spread  avg_price
0     2172.0     2399.0         227.0     2285.5
1     1000.0     1500.0         500.0     1250.0
2      800.0      820.0          20.0      810.0
3     2040.0     2668.0         628.0     2354.0
4      476.0     1043.0         567.0      759.5


In [7]:
# Apply One-Hot Encoding to commodity (crop) and market (mandi)

data = pd.get_dummies(
    data,
    columns=['commodity', 'market'],
    drop_first=True
)

print("Dataset shape after encoding:", data.shape)

print(data.head())

Dataset shape after encoding: (737392, 1612)
           state     district       date  min_price  max_price  modal_price  \
0    Maharashtra       nashik 2023-06-06     2172.0     2399.0       2300.0   
1    Maharashtra       satara 2023-06-06     1000.0     1500.0       1250.0   
2  Uttar Pradesh     mainpuri 2023-06-06      800.0      820.0        810.0   
3      Rajasthan  chittorgarh 2023-06-06     2040.0     2668.0       2300.0   
4      Rajasthan   pratapgarh 2023-06-06      476.0     1043.0        617.0   

   year  month  day  price_spread  ...  market_Williamnagar  \
0  2023      6    6         227.0  ...                False   
1  2023      6    6         500.0  ...                False   
2  2023      6    6          20.0  ...                False   
3  2023      6    6         628.0  ...                False   
4  2023      6    6         567.0  ...                False   

   market_Wokha Town  market_Yamuna Nagar  market_Yawal  market_Yeola  \
0              False        

In [8]:
# Target variable (jo predict karna hai)
y = data['modal_price']

# Features (input variables)
X = data.drop(columns=['modal_price', 'state', 'district', 'date'])

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print(X.head())

Feature shape: (737392, 1608)
Target shape: (737392,)
   min_price  max_price  year  month  day  price_spread  avg_price  \
0     2172.0     2399.0  2023      6    6         227.0     2285.5   
1     1000.0     1500.0  2023      6    6         500.0     1250.0   
2      800.0      820.0  2023      6    6          20.0      810.0   
3     2040.0     2668.0  2023      6    6         628.0     2354.0   
4      476.0     1043.0  2023      6    6         567.0      759.5   

   commodity_Potato  commodity_Rice  commodity_Tomato  ...  \
0             False           False             False  ...   
1             False           False              True  ...   
2              True           False             False  ...   
3             False           False             False  ...   
4             False           False             False  ...   

   market_Williamnagar  market_Wokha Town  market_Yamuna Nagar  market_Yawal  \
0                False              False                False         F

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (589913, 1608)
Testing data shape: (147479, 1608)


In [10]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=20,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Model training completed")

Model training completed


In [11]:
print("Hii")

Hii


In [12]:
from sklearn.metrics import mean_absolute_error

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)

print("Model Error (MAE):", mae)

Model Error (MAE): 35.055341444899824


In [14]:
import joblib

joblib.dump(model, "mandi_price_model.pkl")

print("Model saved successfully")

Model saved successfully


In [15]:
import joblib
import pandas as pd

# model load
model = joblib.load("mandi_price_model.pkl")

# farmer input (example)
crop = "Tomato"
market = "Bhubaneswar"
date = "2026-03-20"

production_cost = 1200
yield_quintals = 20
fuel_cost_per_km = 10

print("Farmer Input Loaded")

Farmer Input Loaded
